# Does TEMPO NO<sub>2</sub> carry information about post-storm nitrogen loading?

**A falsifiable, negative-result-friendly test for one small watershed.**

## The question

For storm events in a small watershed, does **antecedent TEMPO tropospheric NO<sub>2</sub>
column exposure** (1, 3, 7, 14 days before the event) explain any variance in
**event-integrated nitrogen loading** that is *not already explained by precipitation and
antecedent dry days*?

This is a **supervised regression on a handful of events**, which is the kind of problem
TEMPO's ~3-year record is structurally able to support. It is not a forecasting problem.

## The data

| Role | Source | Real or staged? |
| --- | --- | --- |
| Discharge, gauge height, (maybe) nitrate | USGS NWIS instantaneous values | live from source |
| Tidal water level (if the site is tidal) | NOAA CO-OPS | live from source |
| NO<sub>2</sub> tropospheric column | staged TEMPO L3 Zarr (`TEMPO_DATA_URI`) | real, but **temporally limited — the notebook measures how limited** |
| Precipitation | USGS `00045` → NCEI GHCN-Daily → user CSV (tiered, fail-soft) | live from source |

## The baseline

A precipitation-only model: `event precipitation` + `antecedent dry days`.
**TEMPO has to beat that, out of sample, to have earned anything.**

## What would falsify the result

Pre-registered in the cell titled *"Decision rule"*, **before** any model is fit. In short:
TEMPO only "adds information" if it improves leave-one-event-out cross-validated RMSE by
more than 5% *and* survives a permutation test at a Bonferroni-corrected α *and* the events
actually had enough TEMPO coverage to mean anything. Otherwise the conclusion cell prints
that TEMPO added nothing detectable — **which is a publishable result, not a failure.**

## The honest headline, stated up front

The physical chain from *NO<sub>2</sub> column* to *nitrogen in a stream* is long and every
link is loosely constrained (see the **Assumption chain** section). The propagated
uncertainty is one to two orders of magnitude. So this notebook tests for a **statistical
association**, and must not be read as a mass balance. If an association shows up, the
next question is *why*, not *how much*.


---

## Provenance and what has not been verified

This notebook was written on a laptop with **no access to the hub, the TEMPO store, USGS, or
NOAA**. Nothing in it has been executed. Specifically **unverified**:

- **The USGS site numbers in the parameter cell are starting guesses, not facts.** The next
  cell checks them against NWIS and prints the station name, drainage area, and period of
  record. If they fail, there is a bounding-box search cell to find real ones.
- **Column names returned by `sources.read_usgs_instantaneous_values` and
  `sources.read_noaa_coops`.** Every consumer of those frames goes through a normalizer that
  *detects* the columns and prints what it picked, rather than assuming a layout.
- **Coordinate and variable names in the staged TEMPO Zarr.** Same treatment — detected, not
  assumed.
- **The temporal coverage of the staged TEMPO store.** This is the single biggest risk to the
  whole analysis, so it gets its own feasibility gate that runs before any modelling.

Where something could not be checked, the code prints what it found instead of failing
silently. No synthetic data is substituted for real data anywhere in this notebook.


In [ ]:
import json
import math

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from tempo_earth2.config import WorkshopConfig, describe_environment
from tempo_earth2 import tempo, sources, plots

config = WorkshopConfig.from_env()
config.ensure_dirs()

env = describe_environment()
for k, v in env.items():
    print(f"{k:>28}: {v}")

OUTPUTS = config.outputs
print(f"\nwriting results to: {OUTPUTS}")


---

# 1 · Parameters

**Everything you need to change is in the next cell.** Nothing below it is hard-coded.


In [ ]:
# =============================================================================
# ======================  EDIT EVERYTHING IN THIS CELL  =======================
# =============================================================================

# ---- 1.1 The watershed ------------------------------------------------------
# STARTING GUESSES ONLY. The next cell verifies these against NWIS and prints
# the station name, drainage area, and period of record. If they fail, run the
# bounding-box search cell to find real gauges near your site.
USGS_SITES = [
    "01589197",     # candidate: small Chesapeake-margin catchment (UNVERIFIED)
]

# Date range for everything. Keep it inside TEMPO's operational record.
# NOTE: the fetch is automatically extended backwards by max(ANTECEDENT_WINDOWS_DAYS)
# so that the earliest event still has a full antecedent window.
DATE_START = "2024-04-01"
DATE_END   = "2026-06-30"

# Watershed footprint: the box TEMPO pixels are averaged over. Longitudes -180..180.
# A small watershed is 1-3 TEMPO pixels across, which is a real limitation - see
# the limitations section. WEST, SOUTH, EAST, NORTH.
WATERSHED_BBOX = (-76.75, 38.80, -76.45, 39.00)

# The airshed: N deposition to a watershed is sourced from a much larger upwind
# area than the watershed itself. We compute exposure over BOTH footprints.
AIRSHED_BUFFER_DEG = 1.0

# Which TEMPO region to pull from the staged store before subsetting.
# One of tempo.REGIONS: "northeast", "chicago", "losangeles", "texas", "conus".
TEMPO_REGION = "northeast"

# ---- 1.2 Tides --------------------------------------------------------------
# Set SITE_IS_TIDAL = True if the gauge is tidally influenced. Discharge at a
# tidal gauge can be bidirectional and stage is not a runoff proxy - the notebook
# will flag events whose peak coincides with high water.
SITE_IS_TIDAL = False
COOPS_STATION = "8575512"       # NOAA CO-OPS station id (UNVERIFIED). Annapolis, MD.

# ---- 1.3 Storm-event definition (change these; they are the whole ballgame) --
EVENT_METHOD = "rise_rate"      # "rise_rate" or "baseflow"

EVENT_PARAMS = dict(
    # --- shared ---
    min_peak_quantile      = 0.90,   # peak Q must exceed this quantile of the record
    min_separation_hours   = 24.0,   # events closer than this are merged
    max_event_days         = 10.0,   # hard cap on event duration
    recession_fraction     = 0.20,   # event ends when Q falls to pre-event Q + this
                                     # fraction of (peak - pre-event Q)
    # --- rise_rate method ---
    rise_hours             = 6.0,    # window over which the rise is measured
    rise_factor            = 0.50,   # Q must rise by this fraction over rise_hours
    # --- baseflow method (Lyne & Hollick recursive digital filter) ---
    bf_alpha               = 0.98,   # filter parameter, hourly data
    bf_passes              = 3,      # forward / backward / forward
    bf_quickflow_fraction  = 0.30,   # event when quickflow / total exceeds this
)

# ---- 1.4 TEMPO screening ----------------------------------------------------
QA_MAX_FLAG           = 0       # tempo.apply_quality_mask max_flag
MAX_CLOUD_FRACTION    = 0.2     # tempo.apply_quality_mask max_cloud_fraction
MIN_VALID_PIXEL_FRAC  = 0.25    # a scan counts as "seen" if >= this fraction of the
                                # footprint's pixels survive screening

ANTECEDENT_WINDOWS_DAYS = [1, 3, 7, 14]

# TEMPO is a geostationary daylight instrument: it can only observe roughly this
# UTC hour range over North America, and the real range varies with season and
# longitude. The notebook ALSO measures the empirical hour distribution in the
# store and reports both. This is the "what could TEMPO actually see" denominator.
TEMPO_NOMINAL_OBS_HOURS_UTC = (12, 23)
TEMPO_NOMINAL_SCAN_HOURS    = 1.0    # hours of record one scan is taken to represent

# ---- 1.5 Target: nitrogen load, or the stand-in -----------------------------
# Continuous nitrate is NWIS parameter 99133 (nitrate+nitrite as N, in situ).
# 00631 / 00618 are usually discrete lab samples and will normally NOT appear in
# the instantaneous-values service - they are included only so the attempt is logged.
NITRATE_PARAM_CODES = ["99133", "00631", "00618"]

# If no chemistry is found, the notebook falls back to event runoff volume as a
# STAND-IN for load, and says so loudly, everywhere. Set to False to hard-stop
# instead of falling back.
ALLOW_DISCHARGE_STANDIN = True

# ---- 1.6 Precipitation (for the baseline TEMPO has to beat) -----------------
PRECIP_USGS_PARAM   = "00045"   # tier 1: precipitation at the USGS gauge itself
GHCN_STATION_ID     = None      # tier 2: e.g. "USW00013721". None to skip.
PRECIP_CSV_PATH     = None      # tier 3: local CSV with columns time,precip_mm
DRY_DAY_THRESHOLD_MM = 1.0      # a day with less precip than this counts as dry

# ---- 1.7 Pre-registered decision rule ---------------------------------------
MIN_EVENTS_FOR_REGRESSION = 12      # below this the regression is not interpretable
MIN_MEDIAN_WINDOW_COVERAGE = 0.10   # median fraction of the antecedent window with a
                                    # valid TEMPO observation, below which exposure is
                                    # too sparse to mean anything
REQUIRED_RMSE_IMPROVEMENT = 0.05    # 5% relative LOO-CV RMSE improvement
PERMUTATION_ITERATIONS    = 2000
ALPHA_UNCORRECTED         = 0.05    # Bonferroni-corrected by len(ANTECEDENT_WINDOWS_DAYS)

RANDOM_SEED = 20260914

# =============================================================================
# ====================  END OF THE CELL YOU NEED TO EDIT  =====================
# =============================================================================

rng = np.random.default_rng(RANDOM_SEED)

AIRSHED_BBOX = (
    WATERSHED_BBOX[0] - AIRSHED_BUFFER_DEG,
    WATERSHED_BBOX[1] - AIRSHED_BUFFER_DEG,
    WATERSHED_BBOX[2] + AIRSHED_BUFFER_DEG,
    WATERSHED_BBOX[3] + AIRSHED_BUFFER_DEG,
)
MAX_WINDOW_DAYS = max(ANTECEDENT_WINDOWS_DAYS)
FETCH_START = (pd.Timestamp(DATE_START, tz="UTC") - pd.Timedelta(days=MAX_WINDOW_DAYS)).strftime("%Y-%m-%d")
ALPHA_CORRECTED = ALPHA_UNCORRECTED / len(ANTECEDENT_WINDOWS_DAYS)

print(f"sites                : {USGS_SITES}")
print(f"analysis range       : {DATE_START} -> {DATE_END}")
print(f"fetch range          : {FETCH_START} -> {DATE_END}  (backed up {MAX_WINDOW_DAYS} d for antecedent windows)")
print(f"watershed bbox       : {WATERSHED_BBOX}")
print(f"airshed bbox         : {AIRSHED_BBOX}")
print(f"event method         : {EVENT_METHOD}")
print(f"antecedent windows   : {ANTECEDENT_WINDOWS_DAYS} days")
print(f"corrected alpha      : {ALPHA_CORRECTED:.4f}  ({ALPHA_UNCORRECTED} / {len(ANTECEDENT_WINDOWS_DAYS)} windows)")


---

## 1.8 · Frame normalizers

`sources.read_usgs_instantaneous_values` and `sources.read_noaa_coops` return pandas frames
whose exact column layout was **not verifiable when this notebook was written**. Rather than
assume, these helpers *detect* the time / value / site columns and print what they picked.
If detection fails they raise with the real column list, so the fix is a one-liner.


In [ ]:
_TIME_HINTS = ("time_utc", "datetime", "date_time", "dateTime", "time", "t", "date")
_VALUE_HINTS = ("value", "result", "measurement", "v")


def pick_time_column(df, label=""):
    # 1. an actual datetime dtype wins
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            print(f"  [{label}] time column -> {c!r} (datetime dtype)")
            return c
    # 2. otherwise a name hint that parses
    lower = {str(c).lower(): c for c in df.columns}
    for hint in _TIME_HINTS:
        if hint in lower:
            c = lower[hint]
            print(f"  [{label}] time column -> {c!r} (name hint)")
            return c
    raise KeyError(
        f"[{label}] could not find a time column. Columns present: {list(df.columns)}. "
        "Add the right name to _TIME_HINTS."
    )


def pick_value_column(df, parameter_code=None, label=""):
    cols = [str(c) for c in df.columns]
    # 1. a column carrying the parameter code and not a qualifier column
    if parameter_code:
        cands = [c for c in cols if parameter_code in c and not c.endswith("_cd")]
        if cands:
            c = min(cands, key=len)
            print(f"  [{label}] value column -> {c!r} (matched parameter code {parameter_code})")
            return c
    # 2. a generic value-ish name
    lower = {c.lower(): c for c in cols}
    for hint in _VALUE_HINTS:
        if hint in lower:
            print(f"  [{label}] value column -> {lower[hint]!r} (name hint)")
            return lower[hint]
    # 3. the only numeric column that is not a site id
    numeric = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c]) and "site" not in str(c).lower()
    ]
    if len(numeric) == 1:
        print(f"  [{label}] value column -> {numeric[0]!r} (sole numeric column)")
        return numeric[0]
    raise KeyError(
        f"[{label}] could not find a value column for parameter {parameter_code!r}. "
        f"Columns present: {list(df.columns)}."
    )


def pick_site_column(df):
    for c in df.columns:
        if "site" in str(c).lower():
            return c
    return None


def to_series(df, parameter_code=None, site=None, label=""):
    # Normalize any of these frames into a tz-aware, sorted, float pd.Series.
    if df is None or len(df) == 0:
        print(f"  [{label}] EMPTY frame returned")
        return pd.Series(dtype="float64", index=pd.DatetimeIndex([], tz="UTC"))

    work = df.copy()

    # Long/tidy layout: a parameter column plus a single value column.
    param_col = next(
        (c for c in work.columns
         if str(c).lower() in ("parameter_code", "parametercd", "parameter", "variable")),
        None,
    )
    if param_col is not None and parameter_code is not None:
        work = work[work[param_col].astype(str) == str(parameter_code)]
        print(f"  [{label}] filtered long frame on {param_col}=={parameter_code} -> {len(work)} rows")

    site_col = pick_site_column(work)
    if site is not None and site_col is not None:
        work = work[work[site_col].astype(str).str.zfill(8) == str(site).zfill(8)]

    tcol = pick_time_column(work, label=label)
    vcol = pick_value_column(work, parameter_code=parameter_code, label=label)

    idx = pd.to_datetime(work[tcol], utc=True, errors="coerce")
    vals = pd.to_numeric(work[vcol], errors="coerce")
    s = pd.Series(vals.to_numpy(), index=pd.DatetimeIndex(idx), name=label)
    s = s[~s.index.isna()].sort_index()
    s = s[~s.index.duplicated(keep="first")].dropna()
    print(f"  [{label}] -> {len(s)} points, {s.index.min()} .. {s.index.max()}")
    return s


def mpl_time(x):
    # matplotlib + pandas 3.0: pandas no longer auto-registers its matplotlib date
    # converters, and numpy has no tz-aware datetime dtype - so a tz-aware
    # DatetimeIndex reaches matplotlib as an OBJECT array of Timestamps and raises
    #   TypeError: float() argument must be a string or a real number, not 'Timestamp'
    # Every time axis in this notebook is UTC, so dropping tzinfo for PLOTTING ONLY
    # is lossless (the axis labels already say UTC).
    # Use this for axis values only - never for arithmetic.
    if isinstance(x, pd.Timestamp):
        return (x.tz_convert(None) if x.tzinfo is not None else x).to_pydatetime()
    idx = pd.DatetimeIndex(x)
    if idx.tz is not None:
        idx = idx.tz_convert(None)
    return idx.to_numpy()


---

# 2 · Verify the sites actually exist

The USGS site numbers in the parameter cell are guesses. This cell asks NWIS what they are.

It uses the NWIS **site service** directly (`pandas.read_csv` over the public RDB endpoint)
rather than `tempo_earth2.sources`, because the workshop package exposes values, not site
metadata. It is a bounded, single-request, read-only call and it fails soft — if the network
is unavailable it says so and lets you proceed on faith.


In [ ]:
NWIS_SITE_URL = "https://waterservices.usgs.gov/nwis/site/"


def describe_usgs_sites(sites):
    url = (
        f"{NWIS_SITE_URL}?format=rdb&sites={','.join(sites)}"
        "&siteOutput=expanded&siteStatus=all"
    )
    print(f"GET {url}\n")
    try:
        df = pd.read_csv(url, sep="\t", comment="#", dtype=str, skiprows=[1])
    except Exception as exc:
        print(f"!! NWIS site service unreachable or returned nothing: {exc!r}")
        print("   Proceeding without verification - the discharge fetch below will fail")
        print("   loudly if the site numbers are wrong.")
        return None
    keep = [c for c in (
        "site_no", "station_nm", "dec_lat_va", "dec_long_va",
        "drain_area_va", "contrib_drain_area_va", "huc_cd", "site_tp_cd",
    ) if c in df.columns]
    out = df.loc[:, keep]
    print(out.to_string(index=False))
    if "drain_area_va" in out.columns:
        print("\n  drain_area_va is in square miles. 'Small watershed' for this analysis")
        print("  means roughly < 100 sq mi; much larger and the airshed / watershed")
        print("  distinction blurs and travel time smears the event definition.")
    return out


site_info = describe_usgs_sites(USGS_SITES)


In [ ]:
# OPTIONAL - only run this if the cell above could not find your sites.
# Lists every USGS gauge with a discharge record inside the airshed box.
RUN_SITE_SEARCH = False

if RUN_SITE_SEARCH:
    w, s, e, n = AIRSHED_BBOX
    url = (
        f"{NWIS_SITE_URL}?format=rdb"
        f"&bBox={w:.6f},{s:.6f},{e:.6f},{n:.6f}"
        "&siteOutput=expanded&siteStatus=all&hasDataTypeCd=iv&parameterCd=00060"
    )
    print(f"GET {url}\n")
    try:
        found = pd.read_csv(url, sep="\t", comment="#", dtype=str, skiprows=[1])
        keep = [c for c in ("site_no", "station_nm", "dec_lat_va", "dec_long_va",
                            "drain_area_va", "site_tp_cd") if c in found.columns]
        found = found.loc[:, keep]
        if "drain_area_va" in found.columns:
            found = found.assign(
                _area=pd.to_numeric(found["drain_area_va"], errors="coerce")
            ).sort_values("_area").drop(columns="_area")
        print(found.to_string(index=False))
        print(f"\n{len(found)} gauges found. Paste the site_no you want into USGS_SITES above.")
    except Exception as exc:
        print(f"!! site search failed: {exc!r}")
else:
    print("site search skipped (set RUN_SITE_SEARCH = True to use it)")


---

# 3 · Discharge and gauge height

`sources.read_usgs_instantaneous_values(sites, start, end, parameter_codes=...)`.

Requests are **chunked by year**. NWIS will happily refuse or truncate a multi-year
instantaneous-values request for a 15-minute series, and the pod has a 16 GB ceiling.


In [ ]:
def fetch_usgs_chunked(sites, start, end, parameter_codes, label):
    # Year-by-year so no single NWIS request is unbounded.
    start_ts = pd.Timestamp(start, tz="UTC")
    end_ts = pd.Timestamp(end, tz="UTC")
    frames = []
    cursor = start_ts
    while cursor < end_ts:
        stop = min(cursor + pd.DateOffset(years=1), end_ts)
        a, b = cursor.strftime("%Y-%m-%d"), stop.strftime("%Y-%m-%d")
        try:
            chunk = sources.read_usgs_instantaneous_values(
                sites, a, b, parameter_codes=parameter_codes
            )
            n = 0 if chunk is None else len(chunk)
            print(f"  [{label}] {a} .. {b}: {n} rows")
            if n:
                frames.append(chunk)
        except Exception as exc:
            print(f"  [{label}] {a} .. {b}: FAILED {type(exc).__name__}: {exc}")
        cursor = stop
    if not frames:
        return None
    return pd.concat(frames, ignore_index=True)


print("discharge (00060):")
raw_q = fetch_usgs_chunked(USGS_SITES, FETCH_START, DATE_END, "00060", "00060")
if raw_q is None or len(raw_q) == 0:
    raise RuntimeError(
        "No discharge returned for "
        f"{USGS_SITES} over {FETCH_START}..{DATE_END}. Either the site numbers are wrong "
        "(see the verification cell above), the site has no instantaneous discharge record, "
        "or NWIS is unreachable from the pod. Nothing downstream can run without this."
    )
print("\ncolumns returned:", list(raw_q.columns))
raw_q.head()


In [ ]:
print("normalizing discharge:")
q_raw = to_series(raw_q, parameter_code="00060", site=USGS_SITES[0], label="Q_cfs")

# Hourly mean. Instantaneous values are usually 15-minute; hourly is the right
# resolution to pair with TEMPO's hourly-ish scan cadence and keeps memory small.
q = q_raw.resample("1h").mean()
gap_frac = float(q.isna().mean())
print(f"\nhourly series: {len(q)} hours, {gap_frac:.1%} missing")
print(f"range: {q.index.min()} .. {q.index.max()}")
print(f"Q (ft3/s): min={q.min():.3f}  median={q.median():.3f}  max={q.max():.3f}")

# Short gaps are interpolated so the event detector does not fragment; long gaps
# are left as NaN and any event overlapping one is flagged later.
MAX_INTERP_HOURS = 3
q_filled = q.interpolate(limit=MAX_INTERP_HOURS, limit_area="inside")
print(f"after interpolating gaps <= {MAX_INTERP_HOURS} h: {q_filled.isna().mean():.1%} still missing")


In [ ]:
print("gauge height (00065):")
raw_h = fetch_usgs_chunked(USGS_SITES, FETCH_START, DATE_END, "00065", "00065")
if raw_h is not None and len(raw_h):
    h = to_series(raw_h, parameter_code="00065", site=USGS_SITES[0], label="stage_ft").resample("1h").mean()
else:
    h = pd.Series(dtype="float64", index=pd.DatetimeIndex([], tz="UTC"))
    print("  no gauge height at this site - not fatal, it is only used for diagnostics")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]})
axes[0].plot(mpl_time(q_filled.index), q_filled.to_numpy(), lw=0.6, color="#1f77b4")
axes[0].set_yscale("log")
axes[0].set_ylabel("Q (ft$^3$/s, log)")
axes[0].set_title(f"USGS {USGS_SITES[0]} - hourly discharge")
axes[0].grid(alpha=0.3)
if len(h):
    axes[1].plot(mpl_time(h.index), h.to_numpy(), lw=0.6, color="#555555")
    axes[1].set_ylabel("stage (ft)")
else:
    axes[1].text(0.5, 0.5, "no gauge height record", ha="center", va="center",
                 transform=axes[1].transAxes, color="#999999")
axes[1].grid(alpha=0.3)
axes[1].set_xlabel("UTC")
plt.tight_layout()
plt.show()


---

## 3.1 · Tides

If `SITE_IS_TIDAL` is `True` this pulls NOAA CO-OPS water level and flags events whose peak
lands near high water. **At a tidal gauge, a "discharge" record can be bidirectional and stage
is not a runoff proxy** — an event catalog built from either can be picking up the moon rather
than a storm. Requests are chunked by year because CO-OPS bounds the span per call.


In [ ]:
def fetch_coops_chunked(station, start, end, product="water_level", interval="h"):
    start_ts = pd.Timestamp(start, tz="UTC")
    end_ts = pd.Timestamp(end, tz="UTC")
    frames = []
    cursor = start_ts
    while cursor < end_ts:
        stop = min(cursor + pd.DateOffset(years=1), end_ts)
        a, b = cursor.strftime("%Y-%m-%d"), stop.strftime("%Y-%m-%d")
        try:
            chunk = sources.read_noaa_coops(
                station, a, b, product=product, datum="MSL",
                units="metric", interval=interval,
            )
            n = 0 if chunk is None else len(chunk)
            print(f"  [coops] {a} .. {b}: {n} rows")
            if n:
                frames.append(chunk)
        except Exception as exc:
            print(f"  [coops] {a} .. {b}: FAILED {type(exc).__name__}: {exc}")
        cursor = stop
    return pd.concat(frames, ignore_index=True) if frames else None


water_level = pd.Series(dtype="float64", index=pd.DatetimeIndex([], tz="UTC"))
if SITE_IS_TIDAL:
    raw_wl = fetch_coops_chunked(COOPS_STATION, FETCH_START, DATE_END)
    if raw_wl is not None:
        print("\ncolumns returned:", list(raw_wl.columns))
        water_level = to_series(raw_wl, label="water_level_m").resample("1h").mean()
    else:
        print("!! no CO-OPS data returned - tidal flagging will be skipped")
else:
    print("SITE_IS_TIDAL is False - skipping CO-OPS.")
    print("If your gauge is anywhere near the head of tide, set it True and re-run:")
    print("a tidally forced stage record will manufacture 'storm events' twice a day.")


---

# 4 · Storm-event catalog

Two definitions are implemented; `EVENT_METHOD` selects one and every threshold is in
`EVENT_PARAMS`. **The event definition is the most consequential free parameter in this
notebook** — it determines n, and with a sample this small, n determines the answer. Change
it and re-run; if the conclusion flips, that is itself the finding.

- **`rise_rate`** — an event starts when Q rises by `rise_factor` of its own value over
  `rise_hours`. Simple, transparent, no filter assumptions.
- **`baseflow`** — Lyne & Hollick one-parameter recursive digital filter (forward/backward/
  forward passes) separates quickflow; an event is where quickflow fraction exceeds
  `bf_quickflow_fraction`. Standard, but `bf_alpha` is not physically identifiable.

Both then share the same post-processing: merge events closer than `min_separation_hours`,
drop peaks below `min_peak_quantile`, end on recession to `recession_fraction` of the rise,
and cap at `max_event_days`.


In [ ]:
def lyne_hollick(values, alpha=0.98, passes=3):
    # Lyne & Hollick one-parameter recursive digital filter.
    #   f[i]  = alpha*f[i-1] + (1+alpha)/2 * (y[i] - y[i-1])
    #   qf[i] = max(f[i], 0);  bf[i] = y[i] - qf[i]
    # Each extra pass re-filters the PREVIOUS pass's baseflow, alternating
    # direction (forward / backward / forward). Returns baseflow, same length.
    # NaN-safe: gaps are filled for the filter and re-masked at the end.
    x = np.asarray(values, dtype="float64")
    nan_mask = np.isnan(x)
    if nan_mask.all():
        return np.full_like(x, np.nan)
    current = pd.Series(x).ffill().bfill().to_numpy()

    for p in range(passes):
        seq = current if p % 2 == 0 else current[::-1]
        f = np.zeros_like(seq)
        for i in range(1, len(seq)):
            f[i] = alpha * f[i - 1] + 0.5 * (1.0 + alpha) * (seq[i] - seq[i - 1])
        bf = seq - np.maximum(f, 0.0)
        bf = np.clip(bf, 0.0, seq)              # baseflow is non-negative and <= total
        current = bf if p % 2 == 0 else bf[::-1]

    return np.where(nan_mask, np.nan, current)


def _merge_and_trim(intervals, series, params):
    # intervals: list of (start_idx, end_idx). Merge, then trim by peak/duration.
    peak_threshold = float(np.nanquantile(series.to_numpy(), params["min_peak_quantile"]))
    if not intervals:
        return [], peak_threshold
    min_sep = pd.Timedelta(hours=params["min_separation_hours"])
    max_len = pd.Timedelta(days=params["max_event_days"])
    idx = series.index

    merged = [list(intervals[0])]
    for s, e in intervals[1:]:
        if idx[s] - idx[merged[-1][1]] <= min_sep:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])

    out = []
    for s, e in merged:
        seg = series.iloc[s:e + 1]
        if seg.isna().all():
            continue
        peak_pos = int(np.nanargmax(seg.to_numpy()))
        peak_idx = s + peak_pos
        if float(seg.iloc[peak_pos]) < peak_threshold:
            continue
        if idx[e] - idx[s] > max_len:
            e = int(idx.searchsorted(idx[s] + max_len)) - 1
            if e <= peak_idx:
                e = min(peak_idx + 1, len(idx) - 1)
        out.append((s, peak_idx, e))
    return out, peak_threshold


In [ ]:
def detect_events_rise_rate(series, params):
    idx = series.index
    vals = series.to_numpy(dtype="float64")
    step_h = (idx[1] - idx[0]) / pd.Timedelta(hours=1)
    lag = max(1, int(round(params["rise_hours"] / step_h)))

    prior = np.roll(vals, lag)
    prior[:lag] = np.nan
    with np.errstate(invalid="ignore", divide="ignore"):
        rise = (vals - prior) / np.where(prior > 0, prior, np.nan)
    rising = np.nan_to_num(rise, nan=-np.inf) > params["rise_factor"]

    intervals, on = [], None
    for i, r in enumerate(rising):
        if r and on is None:
            on = max(0, i - lag)
        elif not r and on is not None:
            intervals.append((on, i))
            on = None
    if on is not None:
        intervals.append((on, len(vals) - 1))
    return _merge_and_trim(intervals, series, params)


def detect_events_baseflow(series, params):
    vals = series.to_numpy(dtype="float64")
    bf = lyne_hollick(vals, alpha=params["bf_alpha"], passes=params["bf_passes"])
    with np.errstate(invalid="ignore", divide="ignore"):
        qf_frac = (vals - bf) / np.where(vals > 0, vals, np.nan)
    stormy = np.nan_to_num(qf_frac, nan=0.0) > params["bf_quickflow_fraction"]

    intervals, on = [], None
    for i, r in enumerate(stormy):
        if r and on is None:
            on = i
        elif not r and on is not None:
            intervals.append((on, i))
            on = None
    if on is not None:
        intervals.append((on, len(vals) - 1))
    return _merge_and_trim(intervals, series, params)


BASEFLOW = lyne_hollick(
    q_filled.to_numpy(), alpha=EVENT_PARAMS["bf_alpha"], passes=EVENT_PARAMS["bf_passes"]
)
BASEFLOW_S = pd.Series(BASEFLOW, index=q_filled.index, name="baseflow_cfs")

detector = {"rise_rate": detect_events_rise_rate, "baseflow": detect_events_baseflow}[EVENT_METHOD]
triples, peak_threshold = detector(q_filled, EVENT_PARAMS)
print(f"method            : {EVENT_METHOD}")
print(f"peak threshold    : {peak_threshold:.3f} ft3/s (q{EVENT_PARAMS['min_peak_quantile']:.2f} of record)")
print(f"candidate events  : {len(triples)}")


In [ ]:
CFS_TO_M3S = 0.0283168466
SECONDS_PER_HOUR = 3600.0


def build_event_table(series, baseflow, triples, params):
    idx = series.index
    rows = []
    for k, (s, p, e) in enumerate(triples):
        pre_q = float(np.nanmin(series.iloc[max(0, s - 6):s + 1].to_numpy())) if s > 0 else float(series.iloc[s])
        peak_q = float(series.iloc[p])
        rise = peak_q - pre_q

        # End on recession to pre-event + recession_fraction * rise, else keep e.
        end = e
        if rise > 0:
            target = pre_q + params["recession_fraction"] * rise
            tail = series.iloc[p:e + 1].to_numpy()
            below = np.where(tail <= target)[0]
            if below.size:
                end = p + int(below[0])

        seg_q = series.iloc[s:end + 1]
        seg_b = baseflow.iloc[s:end + 1]
        quick = np.clip(seg_q.to_numpy() - seg_b.to_numpy(), 0.0, None)

        rows.append(dict(
            event_id       = k,
            start          = idx[s],
            peak_time      = idx[p],
            end            = idx[end],
            duration_hours = (idx[end] - idx[s]) / pd.Timedelta(hours=1),
            pre_event_q_cfs= pre_q,
            peak_q_cfs     = peak_q,
            # volumes in m3: sum over hourly steps
            volume_m3      = float(np.nansum(seg_q.to_numpy()) * CFS_TO_M3S * SECONDS_PER_HOUR),
            quickflow_m3   = float(np.nansum(quick) * CFS_TO_M3S * SECONDS_PER_HOUR),
            n_missing_hours= int(seg_q.isna().sum()),
        ))
    ev = pd.DataFrame(rows)
    if len(ev):
        ev["quickflow_fraction"] = ev["quickflow_m3"] / ev["volume_m3"].where(ev["volume_m3"] > 0)
    return ev


events = build_event_table(q_filled, BASEFLOW_S, triples, EVENT_PARAMS)

# Only keep events with a full antecedent window inside the fetched record, and
# inside the user's analysis range.
if len(events):
    earliest_ok = q_filled.index.min() + pd.Timedelta(days=MAX_WINDOW_DAYS)
    in_range = (
        (events["start"] >= max(earliest_ok, pd.Timestamp(DATE_START, tz="UTC")))
        & (events["start"] <= pd.Timestamp(DATE_END, tz="UTC"))
    )
    dropped = int((~in_range).sum())
    events = events.loc[in_range].reset_index(drop=True)
    events["event_id"] = np.arange(len(events))
    print(f"dropped {dropped} events without a full {MAX_WINDOW_DAYS}-day antecedent window "
          "or outside the analysis range")

print(f"\nEVENTS: {len(events)}")
if len(events) == 0:
    raise RuntimeError(
        "Zero storm events survived. Loosen EVENT_PARAMS (min_peak_quantile is the usual "
        "culprit), widen the date range, or switch EVENT_METHOD. Nothing below can run."
    )
events


In [ ]:
# Tidal cross-check: does the event peak coincide with high water?
if SITE_IS_TIDAL and len(water_level):
    wl = water_level.reindex(
        water_level.index.union(pd.DatetimeIndex(events["peak_time"]))
    ).interpolate(limit=2)
    hi = float(np.nanquantile(water_level.to_numpy(), 0.85))
    events["water_level_at_peak_m"] = [float(wl.get(t, np.nan)) for t in events["peak_time"]]
    events["peak_near_high_water"] = events["water_level_at_peak_m"] > hi
    n_flag = int(events["peak_near_high_water"].sum())
    print(f"high-water threshold (q0.85): {hi:.3f} m MSL")
    print(f"{n_flag} of {len(events)} event peaks land above it.")
    if n_flag > 0.4 * len(events):
        print("\n!! WARNING: most event peaks coincide with high water. This event catalog")
        print("   is probably detecting the tide, not storms. Move upstream of the head of")
        print("   tide, or detrend the stage record, before believing anything below.")
else:
    events["water_level_at_peak_m"] = np.nan
    events["peak_near_high_water"] = False


In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.plot(mpl_time(q_filled.index), q_filled.to_numpy(), lw=0.6, color="#1f77b4", label="Q")
ax.plot(mpl_time(BASEFLOW_S.index), BASEFLOW_S.to_numpy(), lw=0.7, color="#ff7f0e",
        label=f"baseflow (Lyne-Hollick a={EVENT_PARAMS['bf_alpha']})")
for _, r in events.iterrows():
    ax.axvspan(mpl_time(r["start"]), mpl_time(r["end"]), color="#d62728", alpha=0.18, lw=0)
ax.axhline(peak_threshold, ls="--", lw=0.8, color="#666666",
           label=f"peak threshold (q{EVENT_PARAMS['min_peak_quantile']:.2f})")
ax.set_yscale("log")
ax.set_ylabel("Q (ft$^3$/s, log)")
ax.set_xlabel("UTC")
ax.set_title(f"{len(events)} storm events - method={EVENT_METHOD}")
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---

# 5 · TEMPO: what the store actually contains

**Read this before trusting anything downstream.** The staged workshop TEMPO store is a
*regional, time-limited* Zarr, not the full mission archive. The workshop data catalog
describes one guaranteed high-quality case (six real V04 scans on 31 May 2026) plus a staged
regional NO<sub>2</sub> store whose span is not documented anywhere I could check.

Antecedent exposure over 1/3/7/14-day windows for a multi-year event catalog needs *continuous*
coverage. If the store holds days rather than years, this analysis cannot be done as specified
— and the right response is to say so, not to regress on three points.

So: measure the coverage first, print it, and gate on it.


In [ ]:
tempo_ds = tempo.open_tempo_source(config.tempo_uri)
print(f"TEMPO_DATA_URI = {config.tempo_uri}\n")
print(tempo_ds)


In [ ]:
def pick_dim(ds, candidates, kind):
    names = list(ds.dims) + list(ds.coords)
    lower = {str(n).lower(): n for n in names}
    for c in candidates:
        if c in lower:
            return lower[c]
    raise KeyError(
        f"could not find the {kind} coordinate. dims={list(ds.dims)} coords={list(ds.coords)}. "
        f"Tried {candidates}."
    )


TIME_NAME = pick_dim(tempo_ds, ("time", "scan_time", "t", "valid_time"), "time")
LAT_NAME  = pick_dim(tempo_ds, ("lat", "latitude", "y"), "latitude")
LON_NAME  = pick_dim(tempo_ds, ("lon", "longitude", "x"), "longitude")
print(f"time / lat / lon coordinate names -> {TIME_NAME!r} / {LAT_NAME!r} / {LON_NAME!r}")

no2_var = tempo.NO2_TROPOSPHERIC
print(f"NO2 variable -> {no2_var!r}")
print(f"data variables in store -> {list(tempo_ds.data_vars)}")
for needed in (no2_var, "cloud_fraction", "qa_flag"):
    status = "present" if needed in tempo_ds.variables else "MISSING"
    print(f"  {needed:>16}: {status}")


In [ ]:
scan_times = pd.DatetimeIndex(pd.to_datetime(tempo_ds[TIME_NAME].values, utc=True)).sort_values()
store_days = pd.DatetimeIndex(scan_times.date).unique()

print("=" * 78)
print("TEMPO STORE COVERAGE")
print("=" * 78)
print(f"scans in store        : {len(scan_times)}")
print(f"first scan            : {scan_times.min()}")
print(f"last scan             : {scan_times.max()}")
print(f"distinct calendar days: {len(store_days)}")
span_days = (scan_times.max() - scan_times.min()) / pd.Timedelta(days=1)
print(f"span                  : {span_days:.1f} days")
print(f"days with data / span : {len(store_days)} / {math.ceil(span_days) + 1} "
      f"= {len(store_days) / (math.ceil(span_days) + 1):.1%}")

hours = pd.Series(scan_times.hour).value_counts().sort_index()
print("\nscans per UTC hour (this is TEMPO's real observing window, empirically):")
for hr, n in hours.items():
    print(f"  {hr:02d}Z  {'#' * int(n * 40 / max(hours))} {n}")
EMPIRICAL_OBS_HOURS = sorted(hours.index.tolist())
print(f"\nempirical observing hours (UTC): {EMPIRICAL_OBS_HOURS}")
print(f"nominal assumed in parameters  : {list(range(TEMPO_NOMINAL_OBS_HOURS_UTC[0], TEMPO_NOMINAL_OBS_HOURS_UTC[1] + 1))}")


In [ ]:
# Overlap between the store and the events we need to explain.
need_start = events["start"].min() - pd.Timedelta(days=MAX_WINDOW_DAYS)
need_end = events["start"].max()
overlap = scan_times[(scan_times >= need_start) & (scan_times <= need_end)]

print(f"events span            : {events['start'].min()} .. {events['start'].max()}")
print(f"antecedent coverage need: {need_start} .. {need_end}")
print(f"TEMPO scans in that span: {len(overlap)}")

if len(overlap) == 0:
    print()
    print("!" * 78)
    print("!! THE STAGED TEMPO STORE DOES NOT OVERLAP YOUR EVENT CATALOG AT ALL.")
    print("!! There is no version of this analysis that can run. Options:")
    print("!!   1. Move DATE_START / DATE_END onto the store's actual span printed above.")
    print("!!   2. Use catalog.as_frame('data') to see what else is staged.")
    print("!!   3. Stage more TEMPO yourself with tempo.search_earthdata(...) - this is a")
    print("!!      prep-time operation, not something to do during a 3-day workshop.")
    print("!" * 78)
else:
    print(f"first / last in span    : {overlap.min()} .. {overlap.max()}")


## 5.1 · Screening and the watershed / airshed footprints

Two footprints, because they answer different questions:

- **Watershed box** — the pixels over the catchment itself. Physically the right footprint for
  *local* deposition, but a small watershed is only 1–3 TEMPO pixels across, so this average is
  noisy and partly contaminated by the surrounding landscape inside the same pixel.
- **Airshed box** — the watershed buffered by `AIRSHED_BUFFER_DEG`. Nitrogen deposited on a
  catchment is emitted over a far larger upwind area, so this is arguably the *better* predictor
  even though it is not "the watershed". It is also better sampled.

Screening is mandatory: `apply_quality_mask(max_flag=0, max_cloud_fraction=0.2)`. Unscreened
NO<sub>2</sub> is not a measurement. The surviving-pixel fraction is carried through to the end.


In [ ]:
def spatial_subset(obj, bbox, lat_name=LAT_NAME, lon_name=LON_NAME):
    # Handles ascending or descending coordinate order.
    w, s, e, n = bbox
    lat = obj[lat_name].values
    lon = obj[lon_name].values
    lat_slice = slice(s, n) if lat[0] <= lat[-1] else slice(n, s)
    lon_slice = slice(w, e) if lon[0] <= lon[-1] else slice(e, w)
    return obj.sel({lat_name: lat_slice, lon_name: lon_slice})


# Bound the read: region -> time span we need -> airshed box, all BEFORE compute.
ds = tempo_ds
try:
    ds = tempo.subset(ds, region=TEMPO_REGION)
    print(f"tempo.subset(region={TEMPO_REGION!r}) applied")
except Exception as exc:
    print(f"tempo.subset(region={TEMPO_REGION!r}) failed ({type(exc).__name__}: {exc});")
    print("falling back to the airshed bbox only, which is a tighter subset anyway.")

ds = ds.sel({TIME_NAME: slice(need_start, need_end)})
ds_air = spatial_subset(ds, AIRSHED_BBOX)

n_t = ds_air.sizes.get(TIME_NAME, 0)
n_y = ds_air.sizes.get(LAT_NAME, 0)
n_x = ds_air.sizes.get(LON_NAME, 0)
print(f"\nairshed subset: {n_t} scans x {n_y} lat x {n_x} lon = {n_t * n_y * n_x:,} cells")
if n_y * n_x == 0:
    raise RuntimeError(
        f"The airshed box {AIRSHED_BBOX} selects zero pixels from the store. Check the sign "
        "convention on longitude (this store should be -180..180) and that your watershed is "
        f"inside TEMPO_REGION={TEMPO_REGION!r} whose bounds are in tempo.REGIONS."
    )


In [ ]:
masked_air = tempo.apply_quality_mask(
    ds_air, variable=no2_var,
    max_flag=QA_MAX_FLAG, max_cloud_fraction=MAX_CLOUD_FRACTION,
)
masked_ws = spatial_subset(masked_air, WATERSHED_BBOX)

n_ws = masked_ws.sizes.get(LAT_NAME, 0) * masked_ws.sizes.get(LON_NAME, 0)
print(f"watershed box holds {n_ws} TEMPO pixels "
      f"({masked_ws.sizes.get(LAT_NAME, 0)} x {masked_ws.sizes.get(LON_NAME, 0)})")
if n_ws < 4:
    print("!! Fewer than 4 pixels over the watershed. The watershed-footprint exposure is")
    print("   effectively a single-pixel time series; prefer the airshed predictor and say so.")

spatial_dims = [LAT_NAME, LON_NAME]


def reduce_footprint(da, label):
    mean = da.mean(dim=spatial_dims, skipna=True)
    valid = da.notnull().mean(dim=spatial_dims)
    out = xr.Dataset({f"no2_{label}": mean, f"validfrac_{label}": valid}).compute()
    return out


print("\ncomputing footprint means (this is the only heavy step) ...")
red_air = reduce_footprint(masked_air, "airshed")
red_ws = reduce_footprint(masked_ws, "watershed")

scans = xr.merge([red_air, red_ws]).to_dataframe()
scans.index = pd.DatetimeIndex(pd.to_datetime(scans.index, utc=True))
scans = scans.sort_index()
print(f"scan-level table: {len(scans)} rows")
print("NO2 units are molecules cm-2; typical tropospheric columns are 1e15 - 1e16.")
print(scans.describe().to_string())
scans.head()


### Footprint sanity check

Look at this before trusting any number above. The single most common way to get a silently
wrong answer here is a **bounding box that does not contain the watershed** — a longitude sign
flip, or a catchment that falls outside `TEMPO_REGION`. The box should sit over the catchment,
and the airshed panel should show recognisable NO<sub>2</sub> structure (urban plumes, roads),
not a uniform field.


In [ ]:
# The best-observed scan in the record, as a visual check on the footprints.
if int(scans["validfrac_airshed"].notna().sum()) == 0:
    print("no scans with any valid pixels - nothing to plot")
else:
    best_time = scans["validfrac_airshed"].idxmax()
    best_i = int(np.argmin(np.abs(masked_air[TIME_NAME].values
                                  - np.datetime64(best_time.tz_convert(None)))))
    field = masked_air.isel({TIME_NAME: best_i})

    fig, ax = plt.subplots(figsize=(7.5, 6))
    try:
        plots.plot_no2(
            field,
            title=f"screened TEMPO NO$_2$  {best_time:%Y-%m-%d %H:%MZ}"
                  f"  (valid {scans.loc[best_time, 'validfrac_airshed']:.0%})",
            ax=ax, add_colorbar=True,
        )
    except Exception as exc:
        print(f"plots.plot_no2 failed ({type(exc).__name__}: {exc}); plotting directly")
        field.plot(ax=ax, x=LON_NAME, y=LAT_NAME, cmap="magma_r")

    w, s_, e, n_ = WATERSHED_BBOX
    ax.add_patch(plt.Rectangle((w, s_), e - w, n_ - s_, fill=False,
                               edgecolor="#00ffcc", lw=2.0, zorder=10))
    ax.text(w, n_, " watershed", color="#00ffcc", fontsize=9, va="bottom", zorder=10)
    ax.set_title(ax.get_title(), fontsize=10)
    plt.tight_layout()
    plt.show()

    print(f"airshed box   : {AIRSHED_BBOX}   ({n_y} x {n_x} pixels)")
    print(f"watershed box : {WATERSHED_BBOX}   ({masked_ws.sizes.get(LAT_NAME, 0)} x "
          f"{masked_ws.sizes.get(LON_NAME, 0)} pixels)")
    print("If the teal box is not over your catchment, fix WATERSHED_BBOX and re-run from")
    print("the parameter cell. Every number in this notebook depends on it.")


In [ ]:
# A scan "counts" only if enough of the footprint survived screening.
scans["seen_airshed"] = scans["validfrac_airshed"] >= MIN_VALID_PIXEL_FRAC
scans["seen_watershed"] = scans["validfrac_watershed"] >= MIN_VALID_PIXEL_FRAC

print(f"scans in span                 : {len(scans)}")
print(f"  with >= {MIN_VALID_PIXEL_FRAC:.0%} valid pixels (airshed)  : "
      f"{int(scans['seen_airshed'].sum())} ({scans['seen_airshed'].mean():.1%})")
print(f"  with >= {MIN_VALID_PIXEL_FRAC:.0%} valid pixels (watershed): "
      f"{int(scans['seen_watershed'].sum())} ({scans['seen_watershed'].mean():.1%})")
print()
print("The gap between those two numbers and 100% is cloud and quality screening.")
print("Note the structural problem this creates for THIS question: storms are cloudy,")
print("so TEMPO is systematically blinded in exactly the hours before a storm event.")

fig, ax = plt.subplots(figsize=(15, 3.6))
ok = scans["seen_airshed"]
ax.plot(mpl_time(scans.index[ok]), scans.loc[ok, "no2_airshed"] / 1e15, ".", ms=3,
        color="#2a9d8f", label="airshed, screened")
ax.plot(mpl_time(scans.index[~ok]), scans.loc[~ok, "no2_airshed"] / 1e15, "x", ms=3,
        color="#bbbbbb", label="below valid-pixel threshold")
for _, r in events.iterrows():
    ax.axvline(mpl_time(r["start"]), color="#d62728", lw=0.6, alpha=0.5)
ax.set_ylabel("NO$_2$ (10$^{15}$ molec cm$^{-2}$)")
ax.set_xlabel("UTC")
ax.set_title("Screened TEMPO tropospheric NO$_2$ over the airshed; red lines = storm-event starts")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 5.2 · Antecedent exposure, with the observability denominator

For each event and each window this reports **four different "coverage" numbers**, because
they mean genuinely different things and quoting only the flattering one is how this kind of
analysis goes wrong:

| Column | Meaning |
| --- | --- |
| `cov_of_window` | screened scan-hours ÷ **all** hours in the window. Includes night. **This is the honest headline number and it can never exceed ~50%.** |
| `cov_of_daylight` | screened scan-hours ÷ hours in the window that fall inside TEMPO's empirical observing hours. This is the fraction of *what TEMPO could ever have seen* that it actually saw. |
| `cov_of_scans` | screened scans ÷ scans present in the store for that window. Isolates **cloud/quality loss** from store-gap loss. |
| `day_coverage` | distinct calendar days with ≥1 screened scan ÷ days in the window. Catches the case where all the data is one good day. |

Exposure itself is the **mean of screened scan means** — deliberately *not* a time integral,
because the sampling is irregular and daylight-only, so an "integral" would silently impute
the unobserved 50%+ of the window. A mean of what was seen, reported next to how little was
seen, is the defensible choice.


In [ ]:
OBS_HOURS = set(EMPIRICAL_OBS_HOURS) if len(EMPIRICAL_OBS_HOURS) else set(
    range(TEMPO_NOMINAL_OBS_HOURS_UTC[0], TEMPO_NOMINAL_OBS_HOURS_UTC[1] + 1)
)


def antecedent_exposure(scan_table, event_start, window_days, footprint):
    w0 = event_start - pd.Timedelta(days=window_days)
    win = scan_table.loc[(scan_table.index > w0) & (scan_table.index <= event_start)]
    seen = win.loc[win[f"seen_{footprint}"]]

    window_hours = 24.0 * window_days
    hourly = pd.date_range(w0 + pd.Timedelta(hours=1), event_start, freq="1h")
    daylight_hours = float(sum(1 for t in hourly if t.hour in OBS_HOURS))
    scan_hours = len(seen) * TEMPO_NOMINAL_SCAN_HOURS

    days_in_window = max(1, int(round(window_days)))
    days_seen = len(pd.DatetimeIndex(seen.index.date).unique()) if len(seen) else 0

    return {
        f"no2_{footprint}_{window_days}d":        float(seen[f"no2_{footprint}"].mean()) if len(seen) else np.nan,
        f"no2_{footprint}_{window_days}d_max":    float(seen[f"no2_{footprint}"].max()) if len(seen) else np.nan,
        f"n_scans_{footprint}_{window_days}d":    int(len(win)),
        f"n_seen_{footprint}_{window_days}d":     int(len(seen)),
        f"validfrac_{footprint}_{window_days}d":  float(win[f"validfrac_{footprint}"].mean()) if len(win) else np.nan,
        f"cov_of_window_{footprint}_{window_days}d":   scan_hours / window_hours,
        f"cov_of_daylight_{footprint}_{window_days}d": (scan_hours / daylight_hours) if daylight_hours else np.nan,
        f"cov_of_scans_{footprint}_{window_days}d":    (len(seen) / len(win)) if len(win) else np.nan,
        f"day_coverage_{footprint}_{window_days}d":    days_seen / days_in_window,
    }


rows = []
for _, ev in events.iterrows():
    rec = {"event_id": int(ev["event_id"])}
    for wd in ANTECEDENT_WINDOWS_DAYS:
        for fp in ("airshed", "watershed"):
            rec.update(antecedent_exposure(scans, ev["start"], wd, fp))
    rows.append(rec)

exposure = pd.DataFrame(rows)
events = events.merge(exposure, on="event_id", how="left")
print(f"exposure table: {len(exposure)} events x {exposure.shape[1] - 1} columns")


In [ ]:
# The coverage report - item 3 of the brief, stated plainly.
print("=" * 96)
print("WHAT FRACTION OF EACH ANTECEDENT WINDOW COULD TEMPO ACTUALLY SEE?")
print("(airshed footprint; median over the %d events; see the table above for per-event values)" % len(events))
print("=" * 96)
hdr = f"{'window':>8} | {'cov_of_window':>14} | {'cov_of_daylight':>16} | {'cov_of_scans':>13} | {'day_coverage':>13} | {'events w/ data':>15}"
print(hdr)
print("-" * len(hdr))

coverage_summary = {}
for wd in ANTECEDENT_WINDOWS_DAYS:
    cw = events[f"cov_of_window_airshed_{wd}d"]
    cd = events[f"cov_of_daylight_airshed_{wd}d"]
    cs = events[f"cov_of_scans_airshed_{wd}d"]
    dc = events[f"day_coverage_airshed_{wd}d"]
    have = int(events[f"no2_airshed_{wd}d"].notna().sum())
    coverage_summary[wd] = dict(
        cov_of_window=float(cw.median()), cov_of_daylight=float(cd.median()),
        cov_of_scans=float(cs.median()), day_coverage=float(dc.median()),
        events_with_data=have,
    )
    print(f"{wd:>6}d  | {cw.median():>13.1%} | {cd.median():>15.1%} | "
          f"{cs.median():>12.1%} | {dc.median():>12.1%} | {have:>10} / {len(events)}")

print()
print("Read the first column as the real answer. A geostationary daylight instrument")
print("cannot exceed ~50% of a 24-hour window under any conditions; cloud screening,")
print("quality screening and store gaps take it down from there.")


In [ ]:
# ============================== FEASIBILITY GATE =============================
# Decided BEFORE any model is fit. This does not stop execution - it sets a flag
# that the conclusion cell is required to honour.
n_events = len(events)
median_cov = {wd: coverage_summary[wd]["cov_of_window"] for wd in ANTECEDENT_WINDOWS_DAYS}

gate = {
    "n_events": n_events,
    "n_events_required": MIN_EVENTS_FOR_REGRESSION,
    "n_events_ok": n_events >= MIN_EVENTS_FOR_REGRESSION,
    "median_coverage_by_window": median_cov,
    "coverage_ok": {wd: median_cov[wd] >= MIN_MEDIAN_WINDOW_COVERAGE for wd in ANTECEDENT_WINDOWS_DAYS},
}
gate["any_window_usable"] = any(gate["coverage_ok"].values())
REGRESSION_IS_INTERPRETABLE = bool(gate["n_events_ok"] and gate["any_window_usable"])

print("=" * 78)
print("FEASIBILITY GATE")
print("=" * 78)
print(f"events                    : {n_events} (need >= {MIN_EVENTS_FOR_REGRESSION})  "
      f"{'PASS' if gate['n_events_ok'] else 'FAIL'}")
for wd in ANTECEDENT_WINDOWS_DAYS:
    flag = "PASS" if gate["coverage_ok"][wd] else "FAIL"
    print(f"median coverage, {wd:>2}d window: {median_cov[wd]:6.1%} "
          f"(need >= {MIN_MEDIAN_WINDOW_COVERAGE:.0%})  {flag}")
print()
if REGRESSION_IS_INTERPRETABLE:
    print(">> Gate PASSED. The regression below is interpretable at face value.")
else:
    print(">> Gate FAILED.")
    print(">> The regression below still runs, because seeing the numbers is useful, but")
    print(">> IT IS NOT EVIDENCE EITHER WAY. Do not report a positive result from it, and")
    print(">> do not report 'TEMPO does not help' either - this is an underpowered test,")
    print(">> which is a different finding from a negative one. The conclusion cell says so.")
print("=" * 78)


---

# 6 · Precipitation — the baseline TEMPO has to beat

Tiered and fail-soft, because none of these sources is guaranteed for an arbitrary site:

1. **USGS `00045`** at the gauge itself, through the sanctioned `sources` helper. Best case:
   same request path as discharge, co-located with the watershed. Most gauges don't have it.
2. **NCEI GHCN-Daily** via the public access API, if you set `GHCN_STATION_ID`. Daily totals
   from a nearby met station.
3. **A CSV you supply** at `PRECIP_CSV_PATH` (columns `time`, `precip_mm`).

If all three fail, the baseline degrades to antecedent-dry-days derived from the *discharge*
record, which is a weaker baseline and is flagged as such — TEMPO beating a degraded baseline
would mean less.


In [ ]:
PRECIP_SOURCE = None
precip = pd.Series(dtype="float64", index=pd.DatetimeIndex([], tz="UTC"))

# --- tier 1: USGS 00045 at the gauge -----------------------------------------
try:
    raw_p = fetch_usgs_chunked(USGS_SITES, FETCH_START, DATE_END, PRECIP_USGS_PARAM, PRECIP_USGS_PARAM)
    if raw_p is not None and len(raw_p):
        p_in = to_series(raw_p, parameter_code=PRECIP_USGS_PARAM, site=USGS_SITES[0], label="precip_in")
        # USGS 00045 is incremental precipitation in INCHES per reporting interval.
        precip = (p_in * 25.4).resample("1h").sum()
        PRECIP_SOURCE = f"USGS {PRECIP_USGS_PARAM} at site {USGS_SITES[0]} (inches -> mm)"
except Exception as exc:
    print(f"  tier 1 failed: {type(exc).__name__}: {exc}")

# --- tier 2: NCEI GHCN-Daily --------------------------------------------------
if PRECIP_SOURCE is None and GHCN_STATION_ID:
    url = (
        "https://www.ncei.noaa.gov/access/services/data/v1"
        "?dataset=daily-summaries"
        f"&stations={GHCN_STATION_ID}"
        f"&startDate={FETCH_START}&endDate={DATE_END}"
        "&dataTypes=PRCP&units=metric&format=csv"
    )
    print(f"\nGET {url}")
    try:
        g = pd.read_csv(url)
        g["DATE"] = pd.to_datetime(g["DATE"], utc=True)
        precip = pd.Series(pd.to_numeric(g["PRCP"], errors="coerce").to_numpy(),
                           index=pd.DatetimeIndex(g["DATE"])).sort_index().dropna()
        PRECIP_SOURCE = f"NCEI GHCN-Daily station {GHCN_STATION_ID} (PRCP, mm, DAILY totals)"
    except Exception as exc:
        print(f"  tier 2 failed: {type(exc).__name__}: {exc}")

# --- tier 3: user CSV ---------------------------------------------------------
if PRECIP_SOURCE is None and PRECIP_CSV_PATH:
    try:
        c = pd.read_csv(PRECIP_CSV_PATH)
        tcol = pick_time_column(c, label="precip_csv")
        precip = pd.Series(
            pd.to_numeric(c["precip_mm"], errors="coerce").to_numpy(),
            index=pd.DatetimeIndex(pd.to_datetime(c[tcol], utc=True)),
        ).sort_index().dropna()
        PRECIP_SOURCE = f"user CSV {PRECIP_CSV_PATH}"
    except Exception as exc:
        print(f"  tier 3 failed: {type(exc).__name__}: {exc}")

PRECIP_IS_REAL = PRECIP_SOURCE is not None
print()
if PRECIP_IS_REAL:
    print(f">> precipitation source: {PRECIP_SOURCE}")
    print(f">> {len(precip)} records, {precip.index.min()} .. {precip.index.max()}")
    print(f">> total {precip.sum():.0f} mm over the record")
else:
    print("!" * 78)
    print("!! NO PRECIPITATION RECORD FOUND.")
    print("!! The baseline degrades to a discharge-derived antecedent-wetness proxy, which")
    print("!! is a WEAKER baseline. TEMPO beating a weak baseline is a weaker claim, and")
    print("!! the conclusion cell will say so. Set GHCN_STATION_ID or PRECIP_CSV_PATH.")
    print("!" * 78)


In [ ]:
def event_precip_and_dry_days(event_start, event_end, precip_series, is_real):
    if is_real and len(precip_series):
        during = precip_series.loc[(precip_series.index >= event_start) & (precip_series.index <= event_end)]
        event_mm = float(during.sum())
        daily = precip_series.resample("1D").sum()
        before = daily.loc[daily.index < event_start]
        wet = before.loc[before >= DRY_DAY_THRESHOLD_MM]
        if len(wet):
            dry_days = float((event_start.normalize() - wet.index.max()) / pd.Timedelta(days=1))
        else:
            dry_days = float(len(before))
        ante_7d = float(daily.loc[(daily.index >= event_start - pd.Timedelta(days=7))
                                  & (daily.index < event_start)].sum())
        return event_mm, dry_days, ante_7d, True

    # Degraded proxy from discharge: "dry days" = days since the previous event ended.
    return np.nan, np.nan, np.nan, False


rows = []
prev_end = None
for _, ev in events.iterrows():
    mm, dry, ante7, real = event_precip_and_dry_days(ev["start"], ev["end"], precip, PRECIP_IS_REAL)
    if not real:
        dry = ((ev["start"] - prev_end) / pd.Timedelta(days=1)) if prev_end is not None else np.nan
        mm = float(ev["quickflow_m3"])          # proxy only; flagged below
        ante7 = np.nan
    rows.append(dict(event_id=int(ev["event_id"]),
                     event_precip_mm=mm, antecedent_dry_days=dry, antecedent_precip_7d_mm=ante7))
    prev_end = ev["end"]

events = events.merge(pd.DataFrame(rows), on="event_id", how="left")
print(events[["event_id", "start", "event_precip_mm", "antecedent_dry_days",
              "antecedent_precip_7d_mm"]].to_string(index=False))
if not PRECIP_IS_REAL:
    print("\n!! event_precip_mm above is NOT precipitation - it is quickflow volume (m3),")
    print("!! standing in for it. antecedent_dry_days is days since the previous event.")


---

# 7 · The target: event nitrogen load (or the stand-in)

Continuous nitrate is NWIS parameter **`99133`** (nitrate + nitrite as N, in situ, mg/L as N).
Relatively few gauges have it. `00631` / `00618` are normally *discrete lab samples* served by
the water-quality service, not the instantaneous-values service, so they are attempted and
logged but not expected.

Load is computed as

$$L = \sum_i C_i \, Q_i \, \Delta t$$

with `C` in mg L⁻¹ as N, `Q` in ft³ s⁻¹, converted to **kg N**.

**If no chemistry exists, the target falls back to event runoff volume.** That is a stand-in
for load, not load — it is *by construction* almost perfectly correlated with the precipitation
baseline, which makes TEMPO's job much harder and the test much less meaningful. The flag
follows the result all the way to the conclusion.


In [ ]:
nitrate = pd.Series(dtype="float64", index=pd.DatetimeIndex([], tz="UTC"))
NITRATE_PARAM_USED = None

for code_ in NITRATE_PARAM_CODES:
    print(f"trying nitrate parameter {code_}:")
    try:
        raw_n = fetch_usgs_chunked(USGS_SITES, FETCH_START, DATE_END, code_, code_)
    except Exception as exc:
        print(f"  failed: {type(exc).__name__}: {exc}")
        continue
    if raw_n is not None and len(raw_n):
        try:
            cand = to_series(raw_n, parameter_code=code_, site=USGS_SITES[0], label=f"NO3_{code_}")
        except KeyError as exc:
            print(f"  could not normalize: {exc}")
            continue
        if len(cand) > 0:
            nitrate = cand.resample("1h").mean()
            NITRATE_PARAM_USED = code_
            print(f"  >> using {code_}: {len(nitrate)} hourly values")
            break
    print(f"  no data for {code_}")

HAVE_CHEMISTRY = NITRATE_PARAM_USED is not None


In [ ]:
MG_PER_KG = 1e6
LITRES_PER_CUBIC_FOOT = 28.3168466

if HAVE_CHEMISTRY:
    TARGET_NAME = "event_n_load_kg"
    TARGET_IS_STAND_IN = False
    loads, cover = [], []
    for _, ev in events.iterrows():
        qs = q_filled.loc[ev["start"]:ev["end"]]
        cs = nitrate.reindex(qs.index).interpolate(limit=6, limit_area="inside")
        both = (~qs.isna()) & (~cs.isna())
        # kg = mg/L * ft3/s * L/ft3 * s  / 1e6
        kg = float(np.nansum(
            cs[both].to_numpy() * qs[both].to_numpy() * LITRES_PER_CUBIC_FOOT * SECONDS_PER_HOUR
        ) / MG_PER_KG)
        loads.append(kg)
        cover.append(float(both.sum() / max(1, len(qs))))
    events[TARGET_NAME] = loads
    events["chem_coverage"] = cover
    print(f">> target = event nitrogen load (kg N), from NWIS parameter {NITRATE_PARAM_USED}")
    print(f">> median within-event chemistry coverage: {np.median(cover):.1%}")
    thin = int((np.array(cover) < 0.5).sum())
    if thin:
        print(f"!! {thin} events have < 50% chemistry coverage; their loads are underestimates.")
else:
    if not ALLOW_DISCHARGE_STANDIN:
        raise RuntimeError(
            "No nitrate record at this site and ALLOW_DISCHARGE_STANDIN is False. "
            "Either pick a gauge with parameter 99133, supply your own chemistry, or set "
            "ALLOW_DISCHARGE_STANDIN = True and accept that the target is a stand-in."
        )
    TARGET_NAME = "event_runoff_m3"
    TARGET_IS_STAND_IN = True
    events[TARGET_NAME] = events["volume_m3"]
    events["chem_coverage"] = 0.0
    print("!" * 78)
    print("!!  NO NITROGEN CHEMISTRY AT THIS SITE.")
    print("!!  THE TARGET IS EVENT RUNOFF VOLUME (m3), NOT NITROGEN LOAD.")
    print("!!")
    print("!!  This is a STAND-IN. Everything below is testing whether antecedent NO2")
    print("!!  predicts how much WATER came out, not how much NITROGEN came out. Runoff")
    print("!!  volume is nearly collinear with the precipitation baseline by construction,")
    print("!!  so a null result here is close to uninformative about the real question.")
    print("!!  Do not present a result from this configuration as a nitrogen result.")
    print("!" * 78)

print(f"\nTARGET_NAME       = {TARGET_NAME}")
print(f"TARGET_IS_STAND_IN= {TARGET_IS_STAND_IN}")
print(events[["event_id", "start", TARGET_NAME]].to_string(index=False))


---

# 8 · Decision rule — pre-registered, before any model is fit

Fixing this now is the whole point. With ~10–30 events and four candidate windows it is
trivially easy to find a "significant" TEMPO effect by looking at all of them and reporting
the best one. So:

**TEMPO is declared to add information only if ALL of the following hold:**

1. **Power** — the feasibility gate passed: `n_events ≥ MIN_EVENTS_FOR_REGRESSION`, and the
   window's median coverage ≥ `MIN_MEDIAN_WINDOW_COVERAGE`.
2. **Out-of-sample skill** — leave-one-event-out CV RMSE of `baseline + TEMPO` beats
   `baseline` by more than `REQUIRED_RMSE_IMPROVEMENT` (5%) in relative terms. In-sample R²
   is reported but **carries no weight**: adding any column to an OLS raises it.
3. **Not luck** — a permutation test that shuffles *only* the TEMPO column across events,
   `PERMUTATION_ITERATIONS` times, gives `p < ALPHA_CORRECTED`
   (= 0.05 / number of windows tested, Bonferroni).
4. **Stable sign** — the fitted TEMPO coefficient keeps the same sign in ≥ 80% of LOO folds.

Anything less and the conclusion is *"TEMPO added nothing detectable"* — or, if the gate
failed, *"this test was underpowered"*, which is a different statement and must not be
confused with the first.

**Splits.** Leave-one-event-out is a leave-one-group-out split, which is what the workshop
rules require. A time-ordered 70/30 split is also reported because storm events are
autocorrelated (a wet autumn produces several correlated events) and LOO does not protect
against that. Random row splits are never used.

**Transforms.** Load and runoff are strongly right-skewed, so the target is modelled as
`log`. Exposure is modelled as `log`. Predictors are standardized on the training fold only.


In [ ]:
def ols_fit(X, y):
    A = np.column_stack([np.ones(len(X)), X])
    beta, *_ = np.linalg.lstsq(A, y, rcond=None)
    return beta


def ols_predict(beta, X):
    return np.column_stack([np.ones(len(X)), X]) @ beta


def standardize(train_X, X):
    mu = train_X.mean(axis=0)
    sd = train_X.std(axis=0)
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mu) / sd


def loo_cv(X, y):
    # Leave-one-event-out. Returns predictions and the per-fold coefficient matrix.
    n = len(y)
    preds = np.full(n, np.nan)
    coefs = np.full((n, X.shape[1] + 1), np.nan)
    if n <= X.shape[1] + 2:
        return preds, coefs                      # not enough events to fit honestly
    for i in range(n):
        m = np.ones(n, dtype=bool)
        m[i] = False
        Xtr = standardize(X[m], X[m])
        Xte = standardize(X[m], X[i:i + 1])
        beta = ols_fit(Xtr, y[m])
        preds[i] = ols_predict(beta, Xte)[0]
        coefs[i] = beta
    return preds, coefs


def time_ordered_split_rmse(X, y, train_frac=0.7):
    n = len(y)
    k = int(round(train_frac * n))
    if k <= X.shape[1] + 2 or k >= n:
        return np.nan
    Xtr = standardize(X[:k], X[:k])
    Xte = standardize(X[:k], X[k:])
    beta = ols_fit(Xtr, y[:k])
    return float(np.sqrt(np.mean((y[k:] - ols_predict(beta, Xte)) ** 2)))


def rmse(a, b):
    m = ~(np.isnan(a) | np.isnan(b))
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2))) if m.sum() else np.nan


def r2(y, yhat):
    m = ~(np.isnan(y) | np.isnan(yhat))
    if m.sum() < 3:
        return np.nan
    ss_res = np.sum((y[m] - yhat[m]) ** 2)
    ss_tot = np.sum((y[m] - y[m].mean()) ** 2)
    return float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan


print("OLS / CV helpers defined. Deliberately hand-rolled rather than using")
print("context.linear_baseline(), so that the baseline and the baseline+TEMPO models")
print("see byte-identical folds and standardization - which is what makes the RMSE")
print("difference meaningful. context.linear_baseline() is the workshop's sanctioned")
print("equivalent for single-model fits.")


In [ ]:
BASE_FEATURES = ["log_event_precip", "antecedent_dry_days"]
if PRECIP_IS_REAL:
    BASE_FEATURES.append("log_antecedent_precip_7d")

design = events.copy()
design["log_target"] = np.log(design[TARGET_NAME].clip(lower=1e-9))
design["log_event_precip"] = np.log1p(design["event_precip_mm"].clip(lower=0))
if PRECIP_IS_REAL:
    design["log_antecedent_precip_7d"] = np.log1p(design["antecedent_precip_7d_mm"].clip(lower=0))
for wd in ANTECEDENT_WINDOWS_DAYS:
    for fp in ("airshed", "watershed"):
        design[f"log_no2_{fp}_{wd}d"] = np.log(design[f"no2_{fp}_{wd}d"].clip(lower=1e9))

needed = BASE_FEATURES + ["log_target"]
usable = design.dropna(subset=needed)
print(f"events with a complete baseline + target row: {len(usable)} of {len(design)}")
if len(usable) < 4:
    raise RuntimeError(
        f"Only {len(usable)} events have a complete baseline row. Nothing can be fit. "
        "Most likely cause: no precipitation record (check section 6) or a target full of NaN."
    )
print(f"baseline features: {BASE_FEATURES}")


In [ ]:
def evaluate(frame, base_features, tempo_feature, label):
    sub = frame.dropna(subset=base_features + ["log_target"] + ([tempo_feature] if tempo_feature else []))
    n = len(sub)
    y = sub["log_target"].to_numpy(dtype="float64")
    Xb = sub[base_features].to_numpy(dtype="float64")
    X = np.column_stack([Xb, sub[tempo_feature].to_numpy(dtype="float64")]) if tempo_feature else Xb

    if n <= X.shape[1] + 2:
        return dict(label=label, n=n, insufficient=True)

    # in-sample (reported, not trusted)
    Xs = standardize(X, X)
    beta_full = ols_fit(Xs, y)
    yhat_in = ols_predict(beta_full, Xs)

    preds, coefs = loo_cv(X, y)
    res = dict(
        label=label, n=n, insufficient=False,
        r2_in_sample=r2(y, yhat_in),
        rmse_loo=rmse(y, preds),
        rmse_timesplit=time_ordered_split_rmse(X, y),
        coef=float(beta_full[-1]) if tempo_feature else np.nan,
        y=y, preds=preds, index=sub.index.to_numpy(),
    )
    if tempo_feature:
        signs = coefs[:, -1]
        signs = signs[~np.isnan(signs)]
        res["sign_stability"] = float(np.mean(np.sign(signs) == np.sign(res["coef"]))) if len(signs) else np.nan
    return res


baseline_result = evaluate(usable, BASE_FEATURES, None, "baseline: precip + dry days")
print(f"BASELINE  n={baseline_result['n']}  "
      f"in-sample R2={baseline_result.get('r2_in_sample', float('nan')):.3f}  "
      f"LOO RMSE={baseline_result.get('rmse_loo', float('nan')):.4f}  "
      f"time-split RMSE={baseline_result.get('rmse_timesplit', float('nan')):.4f}")
if baseline_result.get("insufficient"):
    print("!! Not enough events to fit even the baseline honestly. Everything below is void.")


In [ ]:
def permutation_p(frame, base_features, tempo_feature, observed_gain, iters, rng):
    # Shuffle ONLY the TEMPO column across events; recompute the LOO RMSE gain.
    sub = frame.dropna(subset=base_features + ["log_target", tempo_feature])
    y = sub["log_target"].to_numpy(dtype="float64")
    Xb = sub[base_features].to_numpy(dtype="float64")
    t = sub[tempo_feature].to_numpy(dtype="float64")
    if len(y) <= Xb.shape[1] + 3:
        return np.nan, np.array([])
    base_preds, _ = loo_cv(Xb, y)
    base_rmse = rmse(y, base_preds)

    gains = np.empty(iters)
    for k in range(iters):
        tp = rng.permutation(t)
        preds, _ = loo_cv(np.column_stack([Xb, tp]), y)
        gains[k] = (base_rmse - rmse(y, preds)) / base_rmse if base_rmse else np.nan
    p = float((np.sum(gains >= observed_gain) + 1) / (iters + 1))
    return p, gains


results = []
for wd in ANTECEDENT_WINDOWS_DAYS:
    for fp in ("airshed", "watershed"):
        feat = f"log_no2_{fp}_{wd}d"
        r = evaluate(usable, BASE_FEATURES, feat, f"{fp} {wd}d")
        if r.get("insufficient"):
            print(f"{r['label']:>18}: only {r['n']} complete events - skipped")
            results.append(dict(window_days=wd, footprint=fp, feature=feat, n=r["n"],
                                insufficient=True))
            continue
        gain = (baseline_result["rmse_loo"] - r["rmse_loo"]) / baseline_result["rmse_loo"]
        # Permutation is the expensive part; only run it where there is a gain to test.
        if gain > 0:
            p, _ = permutation_p(usable, BASE_FEATURES, feat, gain, PERMUTATION_ITERATIONS, rng)
        else:
            p = 1.0
        results.append(dict(
            window_days=wd, footprint=fp, feature=feat, n=r["n"], insufficient=False,
            r2_in_sample=r["r2_in_sample"], rmse_loo=r["rmse_loo"],
            rmse_timesplit=r["rmse_timesplit"], rmse_gain=gain,
            coef=r["coef"], sign_stability=r.get("sign_stability", np.nan), p_perm=p,
            median_coverage=coverage_summary[wd]["cov_of_window"],
        ))
        print(f"{r['label']:>18}: n={r['n']:>3}  LOO RMSE={r['rmse_loo']:.4f}  "
              f"gain={gain:+.1%}  coef={r['coef']:+.3f}  "
              f"sign-stable={r.get('sign_stability', float('nan')):.0%}  p={p:.4f}")

results = pd.DataFrame(results)


In [ ]:
# Apply the pre-registered rule to every candidate. No post-hoc window picking.
if len(results) and not results["insufficient"].all():
    ok = results.loc[~results["insufficient"]].copy()
    ok["passes_power"]    = REGRESSION_IS_INTERPRETABLE & (ok["median_coverage"] >= MIN_MEDIAN_WINDOW_COVERAGE)
    ok["passes_skill"]    = ok["rmse_gain"] > REQUIRED_RMSE_IMPROVEMENT
    ok["passes_perm"]     = ok["p_perm"] < ALPHA_CORRECTED
    ok["passes_sign"]     = ok["sign_stability"] >= 0.80
    ok["ADDS_INFORMATION"] = (
        ok["passes_power"] & ok["passes_skill"] & ok["passes_perm"] & ok["passes_sign"]
    )
    cols = ["window_days", "footprint", "n", "median_coverage", "r2_in_sample", "rmse_loo",
            "rmse_gain", "coef", "sign_stability", "p_perm",
            "passes_power", "passes_skill", "passes_perm", "passes_sign", "ADDS_INFORMATION"]
    print(ok[cols].to_string(index=False))
    ANY_WINDOW_ADDS = bool(ok["ADDS_INFORMATION"].any())
else:
    ok = results
    ANY_WINDOW_ADDS = False
    print("No window could be evaluated at all.")

print(f"\nBonferroni-corrected alpha applied: {ALPHA_CORRECTED:.4f}")
print(f"ANY_WINDOW_ADDS = {ANY_WINDOW_ADDS}")


In [ ]:
if len(results) and not results["insufficient"].all():
    plot_df = ok.sort_values(["footprint", "window_days"])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for fp, colour in (("airshed", "#2a9d8f"), ("watershed", "#e76f51")):
        sel = plot_df[plot_df["footprint"] == fp]
        axes[0].plot(sel["window_days"], 100 * sel["rmse_gain"], "o-", color=colour, label=fp)
    axes[0].axhline(0, color="#333333", lw=0.8)
    axes[0].axhline(100 * REQUIRED_RMSE_IMPROVEMENT, ls="--", color="#d62728", lw=0.9,
                    label=f"required (+{REQUIRED_RMSE_IMPROVEMENT:.0%})")
    axes[0].set_xlabel("antecedent window (days)")
    axes[0].set_ylabel("LOO-CV RMSE improvement over baseline (%)")
    axes[0].set_title("Does TEMPO help, out of sample?")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    for fp, colour in (("airshed", "#2a9d8f"), ("watershed", "#e76f51")):
        sel = plot_df[plot_df["footprint"] == fp]
        axes[1].plot(sel["window_days"], sel["p_perm"], "o-", color=colour, label=fp)
    axes[1].axhline(ALPHA_CORRECTED, ls="--", color="#d62728", lw=0.9,
                    label=f"corrected alpha = {ALPHA_CORRECTED:.3f}")
    axes[1].set_yscale("log")
    axes[1].set_xlabel("antecedent window (days)")
    axes[1].set_ylabel("permutation p")
    axes[1].set_title("Or is it luck?")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("nothing to plot")


---

# 9 · The assumption chain

Everything above is a **statistical association test**. This section is the reason it can only
ever be that. To read a TEMPO NO<sub>2</sub> column as *nitrogen entering a watershed* you have
to walk five links, and **not one of them is constrained by anything in this notebook.**

The quantitative ranges below are literature-typical order-of-magnitude figures written from
memory, for scale-setting only. **Check each against a current reference before putting any of
them in a proposal or a paper.** They are here to show the *shape* of the uncertainty, not to
be cited.

---

### Link 1 · Tropospheric column → surface concentration

TEMPO retrieves a **vertically integrated** slant-then-vertical column (molecules cm⁻²). The
quantity that matters for deposition is the concentration in the lowest tens of metres.

Converting between them requires the NO<sub>2</sub> **vertical profile shape**, which TEMPO
does not measure — the operational retrieval takes it from a chemical transport model a priori,
and the same a priori is then baked into the air mass factor. So the column and the
column-to-surface factor are not independent.

- Controlled by boundary-layer height, which varies by a factor of ~5–10 between a stable
  pre-dawn inversion and a deep afternoon mixed layer — *within the same day*.
- Column-to-surface relationships derived from satellite–in-situ pairs are commonly reported
  with r in the 0.5–0.8 range and season-dependent slopes.
- **Unconstrained here:** the profile shape, the boundary-layer depth at each scan, and the
  covariance between them and the meteorology that produced the storm.
- **Rough uncertainty: a factor of 2–3.**

The workshop's own `02_tempo_column_vs_surface.ipynb` is exactly this problem, against real EPA
AQS data. **If you care about this link, do that notebook before this one.**

---

### Link 2 · NO<sub>2</sub> → total depositing reactive nitrogen

This is the link most likely to sink the whole idea, and it is not a precision problem — it is
a **wrong-species** problem.

- NO<sub>2</sub> itself deposits comparatively slowly. The dominant **oxidized**-N dry-deposition
  species is **HNO<sub>3</sub>**, plus particulate NO<sub>3</sub>⁻. NO<sub>2</sub> is a
  precursor, not the main depositing form.
- **Reduced nitrogen — NH<sub>3</sub> and NH<sub>4</sub>⁺ — is invisible to TEMPO's
  NO<sub>2</sub> product entirely**, and in the eastern US it is now a large and in many places
  growing share of total N deposition, agricultural areas especially.
- The NO<sub>2</sub> → HNO<sub>3</sub> conversion depends on OH, which depends on photolysis,
  humidity and VOCs — so the conversion efficiency is itself diurnally and seasonally variable.
- **Unconstrained here:** the oxidized/reduced split over this watershed, the NO<sub>2</sub>-to-
  HNO<sub>3</sub> partitioning, and whether the NH<sub>3</sub> field even correlates with the
  NO<sub>2</sub> field locally.
- **Rough uncertainty: a factor of 2–5, plus an unmodelled additive term (reduced N) that can
  be comparable to the whole signal.**

---

### Link 3 · Deposition velocity

Dry deposition flux is `F = V_d × C`. `V_d` is a modelled quantity, never a measured one at
watershed scale.

- Typical `V_d` for NO<sub>2</sub> over vegetation: **~0.1–0.5 cm s⁻¹**, and for HNO<sub>3</sub>
  more like **~1–5 cm s⁻¹** — an order of magnitude apart, so getting the species split wrong
  (Link 2) propagates straight through here.
- `V_d` follows stomatal conductance: diurnal, seasonal, and near-zero for the stomatal pathway
  at night or under drought. It also depends on LAI, canopy wetness, and surface roughness —
  meaning it depends on **land cover**, which varies *within* a single TEMPO pixel over a small
  watershed.
- **Unconstrained here:** all of it. Nothing in this notebook observes canopy state.
- **Rough uncertainty: a factor of 2–5.**

---

### Link 4 · Dry deposition → total atmospheric N input, and the cloud problem

- Total deposition is dry **+ wet**. In the eastern US, wet deposition is frequently comparable
  to or larger than dry, and it is delivered *by the storm itself*.
- **TEMPO cannot see through cloud.** The screening applied in section 5 removes exactly the
  scenes that precede and accompany precipitation. So the instrument is structurally blind
  during the hours that deliver the largest part of the N input for a storm event. The
  `cov_of_scans` column quantified how much was lost — go and look at it again.
- This is not a nuisance; it is a **selection effect aligned with the dependent variable**. If
  antecedent exposure looks predictive, one live alternative explanation is simply that clear
  antecedent skies and the synoptic setup that produces the storm are correlated.
- **Unconstrained here:** the wet deposition term entirely.
- **Rough uncertainty: a factor of 2, and a bias of unknown sign.**

---

### Link 5 · Atmospheric N input → nitrogen in the stream

The last link is ecological and it is the loosest.

- Catchments **retain** most deposited N. Export fractions for forested temperate catchments are
  commonly a small fraction of input — often well under a quarter — and depend on N saturation
  status, soil C:N, stand age, and hydrologic flushing.
- **Legacy and non-atmospheric N usually dominates.** Fertilizer, septic systems, and decades of
  N accumulated in soil and groundwater are the main source of stream nitrate in most settled
  small catchments. The atmospheric increment can be a few percent of export.
- Transit time decouples the timing: N deposited today may leave in this storm, next season, or
  in a decade, depending on the flowpath. **The whole premise that a 1–14 day antecedent window
  is the relevant memory length is an assumption, not a finding** — testing several windows is
  a partial hedge against it, not a fix.
- **Unconstrained here:** retention fraction, source apportionment, transit-time distribution.
- **Rough uncertainty: an order of magnitude, and quite possibly a signal swamped by a much
  larger non-atmospheric term.**

---

### What that adds up to

The next cell multiplies the links out. The point is not the number; the point is that the
number is large enough that **a mechanistic interpretation of any coefficient found above is
not available**, and the only defensible claim is the statistical one:

> *antecedent NO₂ column exposure does / does not carry information about event N loading
> beyond what precipitation already explains, in this catchment, over this period.*

If it does, the interesting follow-up is **why** — and the honest first hypothesis is a
confound (shared synoptic weather driving both NO<sub>2</sub> accumulation and storm type),
not deposition.


In [ ]:
# Explicit uncertainty arithmetic. Ranges are order-of-magnitude, from memory,
# for scale-setting ONLY - verify before citing. See the markdown above.
CHAIN = [
    ("1. column -> surface concentration",       2.0, 3.0,
     "profile shape / BLH not measured; a priori profile also enters the AMF"),
    ("2. NO2 -> total depositing reactive N",    2.0, 5.0,
     "HNO3 dominates oxidized dry dep; NH3/NH4 invisible to TEMPO entirely"),
    ("3. deposition velocity",                   2.0, 5.0,
     "V_d spans ~0.1-0.5 (NO2) vs ~1-5 cm/s (HNO3); stomatal, diurnal, land-cover dependent"),
    ("4. dry -> total (wet) deposition",         2.0, 2.0,
     "wet dep comparable or larger, delivered by the storm; TEMPO is cloud-blinded then"),
    ("5. atmospheric input -> stream export",   10.0, 10.0,
     "catchment retention + legacy/septic/fertilizer N usually dominate; transit time unknown"),
]

print(f"{'link':<42} {'low':>7} {'high':>7}   why it is unconstrained")
print("-" * 118)
lo = hi = 1.0
for name, a, b, why in CHAIN:
    lo *= a
    hi *= b
    print(f"{name:<42} {a:>6.1f}x {b:>6.1f}x   {why}")
print("-" * 118)
print(f"{'MULTIPLICATIVE PRODUCT':<42} {lo:>6.0f}x {hi:>6.0f}x")
print()
print("A TEMPO-derived estimate of watershed N input is uncertain by roughly")
print(f"a factor of {lo:.0f} to {hi:.0f} - that is {math.log10(lo):.1f} to {math.log10(hi):.1f} orders of magnitude.")
print()
print("Consequences, stated plainly:")
print("  * No coefficient above can be read as a deposition rate.")
print("  * A NULL result does not falsify the physics - it is fully consistent with a real")
print("    deposition signal buried under a much larger legacy-N and hydrology signal.")
print("  * A POSITIVE result is not evidence of deposition. Shared synoptic weather driving")
print("    both NO2 accumulation and storm character is the simpler explanation and has to")
print("    be excluded before anything mechanistic is claimed.")
print("  * The way to shorten this chain is to stop trying to close a mass balance and")
print("    instead test a link directly - e.g. column vs. measured surface NO2 (notebook 02),")
print("    or TEMPO against a CASTNET/NADP deposition site.")


---

# 10 · Limitations

Beyond the assumption chain, in rough order of how likely each is to change the answer:

1. **Sample size.** Storm events in a few years of record is `n` in the tens at best. Four
   windows × two footprints is eight tests; Bonferroni is applied, but with small `n` the test
   is underpowered *and* fragile at the same time. A single unusual event can flip it.
2. **The event definition is a free parameter with a large effect on `n`.** Re-run with the
   other `EVENT_METHOD` and with `min_peak_quantile` moved ±0.05. If the conclusion is not
   stable across those, the conclusion is about the detector, not about TEMPO.
3. **Spatial mismatch.** TEMPO pixels are ~2–5 km at nadir and larger at the edges of the FOV.
   A small watershed is 1–3 pixels. The watershed-footprint predictor is close to a single-pixel
   time series and partly measures whatever else is in that pixel.
4. **The airshed box is a crude proxy for a transport footprint.** A real one is wind-direction
   dependent and event-specific. A fixed box will include upwind sources on some days and
   nothing relevant on others. Computing a wind-weighted footprint with
   `forecast.run_forecast(...)` transport winds and `advect.advect(...)` is the obvious upgrade
   and is an afternoon-sized piece of work.
5. **Cloud screening is correlated with the target** (Link 4). This is a selection effect, not
   noise, and no amount of `n` fixes it.
6. **Daylight-only sampling.** Even at perfect quality, coverage of a 24-hour window cannot
   exceed ~50%. The `cov_of_window` column is the honest number; anything that quotes
   `cov_of_scans` alone is flattering itself.
7. **Load computation.** Where chemistry exists, hourly interpolation of nitrate across an event
   is standard but underestimates flashy concentration peaks; the `chem_coverage` column says
   how much of each event was actually observed.
8. **If the target is the discharge stand-in**, the test is close to circular — runoff volume is
   near-collinear with the precipitation baseline by construction. Treat any result from that
   configuration as a plumbing check, not science.
9. **No transport, no chemistry, no deposition model.** This notebook does not advect, does not
   age air masses, and does not model loss. A column measured over the watershed on day −7 may
   have had nothing to do with the air that deposited there.
10. **One catchment.** Even a clean positive result is `n=1` at the catchment level. The design
    that would actually be convincing is many catchments with a leave-one-catchment-out split —
    the pattern in `04_wetland_response.ipynb`.

### If this comes out null — and it probably will

That is a **reportable result**, and it is the reason the notebook was built this way: *"the
TEMPO NO₂ column adds no detectable information about post-storm N loading in catchment X
beyond precipitation, at n events, with this much observability"* is a useful constraint on
where the LEDT should and should not be pointing its effort. Note the observability numbers
alongside it, because *"there was nothing to find"* and *"we could not have found it"* are
different claims and only the coverage table tells them apart.


In [ ]:
# ============================== CONCLUSION ===================================
# Mechanical application of the rule pre-registered in section 8. No judgement calls.
line = "=" * 78
print(line)
print("CONCLUSION")
print(line)
print(f"catchment            : USGS {', '.join(USGS_SITES)}")
print(f"period               : {DATE_START} .. {DATE_END}")
print(f"storm events         : {len(events)}  (method={EVENT_METHOD})")
print(f"target               : {TARGET_NAME}" + ("   <-- STAND-IN, NOT NITROGEN" if TARGET_IS_STAND_IN else ""))
print(f"baseline             : {BASE_FEATURES}" + ("" if PRECIP_IS_REAL else "   <-- DEGRADED, no precipitation record"))
print(f"TEMPO windows tested : {ANTECEDENT_WINDOWS_DAYS} days x (airshed, watershed)")
print("median window coverage: " + ", ".join(
    f"{wd}d={coverage_summary[wd]['cov_of_window']:.1%}" for wd in ANTECEDENT_WINDOWS_DAYS))
print(line)

if not REGRESSION_IS_INTERPRETABLE:
    print("VERDICT: UNDERPOWERED - NO CONCLUSION EITHER WAY.")
    print()
    print("The feasibility gate failed, so this run cannot distinguish 'TEMPO does not help'")
    print("from 'we could not have detected it if it did'. Reasons:")
    if not gate["n_events_ok"]:
        print(f"  - only {gate['n_events']} events, need >= {MIN_EVENTS_FOR_REGRESSION}")
    for wd in ANTECEDENT_WINDOWS_DAYS:
        if not gate["coverage_ok"][wd]:
            print(f"  - {wd}d window: median coverage {median_cov[wd]:.1%} "
                  f"< {MIN_MEDIAN_WINDOW_COVERAGE:.0%} required")
    print()
    print("Do not report this as a negative result. Report it as a coverage finding:")
    print("the staged TEMPO record and/or this catchment's event count are not sufficient")
    print("to test the question as posed.")

elif not ANY_WINDOW_ADDS:
    print("VERDICT: TEMPO ADDS NOTHING OVER THE PRECIPITATION-ONLY BASELINE.")
    print()
    best = ok.loc[ok["rmse_gain"].idxmax()]
    print(f"Best of the {len(ok)} candidates was {best['footprint']} / {best['window_days']}d:")
    print(f"  out-of-sample RMSE change vs baseline : {best['rmse_gain']:+.1%} "
          f"(needed > +{REQUIRED_RMSE_IMPROVEMENT:.0%})")
    print(f"  permutation p                         : {best['p_perm']:.4f} "
          f"(needed < {ALPHA_CORRECTED:.4f})")
    print(f"  coefficient sign stability across LOO : {best['sign_stability']:.0%} (needed >= 80%)")
    print(f"  in-sample R2 (NOT evidence)           : {best['r2_in_sample']:.3f}")
    print()
    print("This is a real, reportable negative result for this catchment and period.")
    print("It does NOT falsify atmospheric N deposition - see the assumption chain: a true")
    print("deposition signal is expected to be small relative to legacy N and hydrology,")
    print("and TEMPO is cloud-blinded in exactly the hours that matter most.")
    if TARGET_IS_STAND_IN:
        print()
        print("!! AND the target was runoff volume, not nitrogen. This null is close to")
        print("!! uninformative about the actual question. Find a gauge with parameter 99133.")

else:
    winners = ok.loc[ok["ADDS_INFORMATION"]]
    print("VERDICT: TEMPO ADDS INFORMATION beyond the precipitation baseline, in "
          f"{len(winners)} of {len(ok)} configurations.")
    print()
    print(winners[["window_days", "footprint", "n", "rmse_gain", "coef",
                   "sign_stability", "p_perm"]].to_string(index=False))
    print()
    print("BEFORE BELIEVING THIS, in order:")
    print("  1. Re-run with the other EVENT_METHOD and with min_peak_quantile +/- 0.05.")
    print("     If the result does not survive, it is a property of the event detector.")
    print("  2. Check the sign. Higher antecedent NO2 should mean MORE N load. A negative")
    print("     coefficient almost certainly means you have found the cloud confound:")
    print("     clear (high-NO2-visibility) antecedent periods preceding particular storms.")
    print("  3. Regress the TEMPO predictor on the baseline predictors. If it is largely")
    print("     explained by them, the 'gain' is shared variance, not new information.")
    print("  4. Repeat on a second, independent catchment. n=1 catchment is not a result.")
    if TARGET_IS_STAND_IN:
        print()
        print("!! The target was runoff volume, not nitrogen. A positive result here says")
        print("!! antecedent NO2 predicts how much WATER came out - which is almost")
        print("!! certainly a weather confound. Do not report this as a nitrogen result.")

print(line)


In [ ]:
# Persist everything to the persistent home so a restart does not lose it.
stamp = pd.Timestamp.utcnow().strftime("%Y%m%dT%H%M%SZ")
outdir = OUTPUTS / f"storm_nitrogen_tempo_{USGS_SITES[0]}_{stamp}"
outdir.mkdir(parents=True, exist_ok=True)

events.to_csv(outdir / "events.csv", index=False)
scans.to_csv(outdir / "tempo_scans.csv")
if len(results):
    results.to_csv(outdir / "regression_results.csv", index=False)

summary = dict(
    sites=USGS_SITES, date_start=DATE_START, date_end=DATE_END,
    watershed_bbox=list(WATERSHED_BBOX), airshed_bbox=list(AIRSHED_BBOX),
    tempo_region=TEMPO_REGION, tempo_uri=str(config.tempo_uri),
    event_method=EVENT_METHOD, event_params=EVENT_PARAMS,
    n_events=int(len(events)),
    qa_max_flag=QA_MAX_FLAG, max_cloud_fraction=MAX_CLOUD_FRACTION,
    min_valid_pixel_frac=MIN_VALID_PIXEL_FRAC,
    antecedent_windows_days=ANTECEDENT_WINDOWS_DAYS,
    coverage_summary={str(k): v for k, v in coverage_summary.items()},
    target=TARGET_NAME, target_is_stand_in=bool(TARGET_IS_STAND_IN),
    nitrate_parameter=NITRATE_PARAM_USED,
    precip_source=PRECIP_SOURCE, precip_is_real=bool(PRECIP_IS_REAL),
    baseline_features=BASE_FEATURES,
    baseline_rmse_loo=float(baseline_result.get("rmse_loo", float("nan"))),
    alpha_corrected=ALPHA_CORRECTED,
    regression_is_interpretable=bool(REGRESSION_IS_INTERPRETABLE),
    any_window_adds=bool(ANY_WINDOW_ADDS),
    random_seed=RANDOM_SEED,
    tempo_store_first_scan=str(scan_times.min()), tempo_store_last_scan=str(scan_times.max()),
    tempo_store_n_scans=int(len(scan_times)),
)
(outdir / "summary.json").write_text(json.dumps(summary, indent=2, default=str))

print(f"wrote:\n  {outdir}/events.csv\n  {outdir}/tempo_scans.csv"
      f"\n  {outdir}/regression_results.csv\n  {outdir}/summary.json")
print("\nsummary.json is the thing to paste into a Slack thread or a demo slide - it carries")
print("the coverage numbers and the stand-in / degraded-baseline flags alongside the verdict,")
print("which is what stops the result being over-read later.")
